In [3]:
import pickle

file_path1 = "/data/huangliqi/demo3/demos/replay/PlaceSphere_DEMO3/motionplanning/20260111_192832.rgb.pd_ee_delta_pose.physx_cpu.pkl"
file_path2 = "/hard_data/user_dataset/huangliqi_dataset/demo3/demos/ms-stack-cube-semi/ms-stack-cube-semi_trajectories_100.pkl"
with open(file_path1, "rb") as f:
    trajectories1 = pickle.load(f)
with open(file_path2, "rb") as f:
    trajectories2 = pickle.load(f)

In [4]:
def show(trajectories):
    print(f"包含轨迹数量: {len(trajectories)}")

    # 查看第一条轨迹的长度
    first_traj = trajectories[0]
    print(f"步数 (T): {len(first_traj['next_observations'])}")

    # 获取第一条轨迹的所有动作
    actions = first_traj['actions'] 
    print(actions)
    print([action.shape for action in actions])
    # 注意：actions[0] 是 NaN
    # 展示图片
    # import matplotlib.pyplot as plt
    # for i in range(len(first_traj['next_observations'])):
    #     plt.imshow(first_traj['next_observations'][i]['rgb_base'].permute(1,2,0))
    #     plt.show()
    #     if i>10:
    #         break
    # print(first_traj['next_observations'][0])
    # print(first_traj['infos'])
    # print([first_traj['infos'][i+1]['elapsed_steps'] for i in range(len(first_traj['infos'])-1)])

show(trajectories1)
# show(trajectories2)


包含轨迹数量: 19
步数 (T): 134
[tensor([nan, nan, nan, nan, nan, nan, nan]), tensor([ 9.4064e-07, -1.2293e-06,  3.4273e-06, -0.0000e+00,  0.0000e+00,
        -0.0000e+00,  1.0000e+00]), tensor([-9.8373e-04, -4.9811e-04, -1.4159e-03, -4.8743e-04,  9.3739e-05,
         1.1950e-02,  1.0000e+00]), tensor([-3.7033e-03, -1.9035e-03, -5.3750e-03, -2.0161e-03, -2.6567e-05,
         4.9754e-02,  1.0000e+00]), tensor([-7.5838e-03, -3.9549e-03, -1.1072e-02, -4.0448e-03, -8.0307e-04,
         1.0022e-01,  1.0000e+00]), tensor([-0.0121, -0.0063, -0.0177, -0.0063, -0.0017,  0.1576,  1.0000]), tensor([-0.0170, -0.0087, -0.0247, -0.0088, -0.0023,  0.2198,  1.0000]), tensor([-0.0221, -0.0110, -0.0318, -0.0114, -0.0026,  0.2851,  1.0000]), tensor([-0.0273, -0.0131, -0.0390, -0.0141, -0.0025,  0.3518,  1.0000]), tensor([-0.0326, -0.0148, -0.0461, -0.0167, -0.0021,  0.4194,  1.0000]), tensor([-0.0380, -0.0163, -0.0532, -0.0194, -0.0016,  0.4874,  1.0000]), tensor([-4.3469e-02, -1.7346e-02, -6.0249e-02, -2.2059e-0

In [ ]:
# 读取h5文件
import h5py
import matplotlib.pyplot as plt
with h5py.File('/data/huangliqi/demo3/demos/StackCube_DEMO3/motionplanning/20251231_181659.rgb.pd_ee_delta_pose.physx_cpu.h5', 'r') as f:
    data_h5 = f['traj_0']
    print(data_h5['actions'][1])
    print(data_h5.keys())
    print(data_h5['rewards'][:])
    print(data_h5['obs'].keys())
    print(data_h5['obs']['agent'].keys())
    print((data_h5['obs']['sensor_data'].keys()))
    print(data_h5['obs']['sensor_data']['base_camera']['rgb'].shape)
    plt.imshow(data_h5['obs']['sensor_data']['base_camera']['rgb'][0])
    plt.show()
    plt.imshow(data_h5['obs']['sensor_data']['hand_camera']['rgb'][0])
    plt.show()
    plt.imshow(data_h5['obs']['sensor_data']['ext_camera']['rgb'][0])
    plt.show()

In [ ]:
import pickle
import numpy as np

def flatten_structure(data, parent_key='', sep=' -> '):
    """
    递归解析字典，将嵌套结构扁平化为 路径: 信息 的映射
    例如: {'obs': {'rgb': arr}} -> {'obs -> rgb': 'float32, (128, 128, 3)'}
    """
    items = {}
    if isinstance(data, dict):
        for k, v in data.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            items.update(flatten_structure(v, new_key, sep=sep))
    elif isinstance(data, list):
        # 针对列表，记录长度并解析第一个元素作为样板
        new_key = f"{parent_key} (list, len={len(data)})"
        if len(data) > 0:
            items.update(flatten_structure(data[0], f"{parent_key}[0]", sep=sep))
        else:
            items[parent_key] = "Empty List"
    elif hasattr(data, 'shape'):
        items[parent_key] = f"{data.dtype}, shape={data.shape}"
    else:
        items[parent_key] = f"{type(data).__name__}"
    return items

def compare_nested_pkls(path1, path2):
    with open(path1, "rb") as f:
        data1 = pickle.load(f)[0] # 取第一个元素
    with open(path2, "rb") as f:
        data2 = pickle.load(f)[0] # 取第一个元素

    struct1 = flatten_structure(data1)
    struct2 = flatten_structure(data2)

    all_keys = sorted(set(struct1.keys()) | set(struct2.keys()))

    print(f"{'Data Path':<60} | {'File 1 Value':<30} | {'File 2 Value':<30}")
    print("-" * 125)

    for key in all_keys:
        val1 = struct1.get(key, "MISSING")
        val2 = struct2.get(key, "MISSING")
        
        status = ""
        if val1 == "MISSING" or val2 == "MISSING":
            status = "❌ KEY MISSING"
        elif val1 != val2:
            status = "⚠️ SHAPE/TYPE DIFF"
        
        print(f"{key:<60} | {str(val1):<30} | {str(val2):<30} {status}")

# 填入你的路径
file_path1 = "/data/huangliqi/demo3/demos/replay/PlaceSphere_DEMO3/motionplanning/20260111_192832.rgb.pd_ee_delta_pose.physx_cpu.pkl"
file_path2 = "/hard_data/user_dataset/huangliqi_dataset/demo3/demos/ms-stack-cube-semi/ms-stack-cube-semi_trajectories_100.pkl"

compare_nested_pkls(file_path1, file_path2)